# Music Sentiment as an Economic Signal

**Working research question:**  
Can shifts in popular music — especially sonic sentiment and genre composition — provide information that improves forecasts of future consumer sentiment?

### Current notebook status
This notebook is currently focused on:

1. Loading and auditing the Billboard Hot 100 data.
2. Loading the 550K Spotify-derived song metadata without the very large lyrics column.
3. Documenting known data-quality problems in the 550K dataset.
4. Engineering a more defensible broad genre taxonomy.
5. Preparing for Billboard-to-song matching.

### Important data decision
The 550K dataset's `year` field is **not treated as the authoritative timeline** because obvious date errors were found (for example, modern songs coded as 1900). The Billboard `Date` column will be the time/popularity backbone for the forecasting analysis.

The original notebook has been cleaned up, but exploratory code that helped reach these decisions is preserved near the bottom as commented-out "thought process" cells.

## 1. Imports and project paths

Only packages needed for the current data-preparation stage are imported below.

Packages for fuzzy matching, forecasting, diagnostics, and plotting are preserved as comments for later so the notebook does not load unnecessary dependencies yet.

In [1]:
import ast
import re
import unicodedata
from pathlib import Path

import pandas as pd

# --- Later stages: keep for reference, but not needed yet ---
# import numpy as np
# import matplotlib.pyplot as plt
#
# from rapidfuzz import fuzz, process
# from sklearn.metrics import mean_squared_error, mean_absolute_error
# from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
# from statsmodels.tsa.ar_model import AutoReg
# from statsmodels.tsa.statespace.sarimax import SARIMAX
# from statsmodels.stats.diagnostic import acorr_ljungbox

PROJECT_ROOT = Path.cwd()

# If the notebook is opened from a notebooks/ subfolder, move one level up.
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

BILLBOARD_PATH = DATA_RAW / "hot100.csv"
SONGS_PATH = DATA_RAW / "songs.csv"
SONG_METADATA_PATH = DATA_PROCESSED / "song_metadata.parquet"

print("Project root:", PROJECT_ROOT)

Project root: /Users/mercyeme/Documents/Data_Projects_2026/RecessionPop


## 2. Billboard Hot 100

The full Billboard table remains untouched as the weekly popularity/time table.

A separate `billboard_songs` table contains only unique `Song` + `Artist` pairs.  
This prevents repeatedly matching the same recording for every week it appears on the chart.

In [2]:
billboard = pd.read_csv(BILLBOARD_PATH)

print("Shape:", billboard.shape)
print("Columns:", billboard.columns.tolist())

billboard["Date"] = pd.to_datetime(billboard["Date"])

display(billboard.head())

print("First chart date:", billboard["Date"].min())
print("Last chart date:", billboard["Date"].max())

Shape: (354858, 8)
Columns: ['Date', 'Song', 'Artist', 'Rank', 'Last Week', 'Peak Position', 'Weeks in Charts', 'Image URL']


,Date,Song,Artist,Rank,Last Week,Peak Position,Weeks in Charts,Image URL
0,1958-07-30,Poor Little Fool,Ricky Nelson,1,-,1,1,NaN
1,1958-07-30,Patricia,Perez Prado And His Orchestra,2,-,2,1,NaN
2,1958-07-30,Splish Splash,Bobby Darin,3,-,3,1,NaN
3,1958-07-30,Hard Headed Woman,Elvis Presley With The Jordanaires,4,-,4,1,https://charts-static.billboard.com/img/1958/0...
4,1958-07-30,When,Kalin Twins,5,-,5,1,NaN


First chart date: 1958-07-30 00:00:00
Last chart date: 2026-07-29 00:00:00


In [3]:
billboard_songs = (
    billboard[["Song", "Artist"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Weekly Billboard rows: {len(billboard):,}")
print(f"Unique song-artist pairs: {len(billboard_songs):,}")

display(billboard_songs.head())

Weekly Billboard rows: 354,858
Unique song-artist pairs: 32,592


,Song,Artist
0,Poor Little Fool,Ricky Nelson
1,Patricia,Perez Prado And His Orchestra
2,Splish Splash,Bobby Darin
3,Hard Headed Woman,Elvis Presley With The Jordanaires
4,When,Kalin Twins


## 3. Load lightweight 550K song metadata

The original `songs.csv` is very large because it contains full lyrics.

For the first pass, only metadata and audio-feature columns are loaded. If the processed Parquet file already exists, the notebook loads that instead of scanning the full CSV again.

**Lyrics stay out of memory until after Billboard matching.**

In [4]:
song_cols = [
    "id",
    "name",
    "album_name",
    "artists",
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_ms",
    "year",
    "genre",
    "popularity",
    "total_artist_followers",
    "avg_artist_popularity",
    "artist_ids",
    "niche_genres",
]

In [5]:
if SONG_METADATA_PATH.exists():
    print("Loading processed metadata:", SONG_METADATA_PATH)
    songs = pd.read_parquet(SONG_METADATA_PATH)

else:
    print("Processed metadata not found; reading songs.csv in chunks.")

    chunks = []

    for chunk in pd.read_csv(
        SONGS_PATH,
        usecols=song_cols,
        chunksize=50_000,
    ):
        chunks.append(chunk)

    songs = pd.concat(chunks, ignore_index=True)

    songs.to_parquet(
        SONG_METADATA_PATH,
        index=False,
    )

print("Songs shape:", songs.shape)
display(songs.head())

Loading processed metadata: /Users/mercyeme/Documents/Data_Projects_2026/RecessionPop/data/processed/song_metadata.parquet
Songs shape: (550622, 23)


,id,name,album_name,artists,danceability,energy,key,loudness,mode,speechiness,...,valence,tempo,duration_ms,year,genre,popularity,total_artist_followers,avg_artist_popularity,artist_ids,niche_genres
0,0Prct5TDjAnEgIqbxcldY9,!,UNDEN!ABLE,"[""HELLYEAH""]",0.415,0.605,7,-11.157,1,0.0575,...,0.193,100.059,79500,2016,Rock,0,769490,52.0,"[""4hxDvVq5t8ebPYPdBl1F9f""]","[""groove metal"", ""metal""]"
1,2ASl4wirkeYm3OWZxXKYuq,!!,Childhood Dreams,"[""Yxngxr1""]",0.788,0.648,7,-9.135,0,0.3150,...,0.287,79.998,114000,2019,Hip-Hop,29,143628,45.0,"[""2jwRHcdgkRhelYEMqndDKe""]",[]
2,5tA3ImW310llKo8EMBj2Ga,!!Noble Stabbings!!,Situationist Comedy,"[""Dillinger Four""]",0.171,0.957,2,-5.749,1,0.1490,...,0.349,175.317,197400,2002,Rock,0,36619,35.0,"[""4YAN46l70QV0PGXlMg0iHi""]","[""melodic hardcore"", ""pop punk"", ""punk"", ""skat..."
3,0fROT4kK5oTm8xO8PX6EJF,!I'll Be Back!,!I'll Be Back!,"[""Ril\u00e8s""]",0.823,0.612,1,-7.767,1,0.2480,...,0.688,142.959,178533,2018,Hip-Hop,43,929303,63.0,"[""6pdcQa7by8IKuoVXvgknlI""]","[""french rap""]"
4,1xBFhv5faebv3mmwxx7DnS,!Lost!,!Lost!,"[""Ril\u00e8s""]",0.729,0.552,7,-8.562,0,0.0650,...,0.380,86.103,186197,2018,Hip-Hop,0,929303,63.0,"[""6pdcQa7by8IKuoVXvgknlI""]","[""french rap""]"


## 4. Data-quality notes

Two findings from exploration matter for the rest of the project:

- `year` contains obvious errors, including modern recordings assigned to 1900.
- The broad `genre` field is useful as a baseline, but some categories — especially `Folk` — contain systematic misclassifications such as reggae/dancehall.

Therefore:

- **Billboard `Date` controls time.**
- **Official broad genre is the fallback classification, not unquestioned ground truth.**
- `niche_genres` supplies evidence for targeted overrides.

In [6]:
print("550K year range:", songs["year"].min(), "to", songs["year"].max())
print("\nOfficial broad genre counts:")
display(songs["genre"].value_counts(dropna=False))

550K year range: 1900 to 2025

Official broad genre counts:


genre
Rock          197168
Pop            72539
Electronic     69755
Folk           50843
Country        46076
Hip-Hop        41745
R&B            27337
Jazz           17010
Blues          15976
Classical      12173
Name: count, dtype: int64

## 5. Parse niche genres

`niche_genres` is stored as a string representation of a Python list.  
Convert it once into an actual list so each tag can be inspected and scored safely.

In [7]:
def parse_genre_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []

    except (ValueError, SyntaxError, TypeError):
        return []


songs["niche_genre_list"] = (
    songs["niche_genres"]
    .apply(parse_genre_list)
)

## 6. Final broad genre taxonomy

The current working taxonomy is intentionally broad enough for monthly genre-share features:

- Pop
- Indie
- Rock
- Hip-Hop
- R&B/Soul
- Country
- Electronic/Dance
- Folk
- Reggae/Dancehall
- Latin/Tropical
- Afrobeats/African Pop
- Jazz
- Classical/Musical
- Blues
- Other

### Classification philosophy

This is **not** a manual remapping of all 754 niche tags.

Instead:

1. Start with the dataset's official broad genre.
2. Rename broad labels where needed (`R&B` → `R&B/Soul`, etc.).
3. Let high-confidence niche tags vote for a corrected category.
4. If there is one clear winning category, use it as an override.
5. If niche evidence is absent or ambiguous, keep the mapped official genre.
6. `Other` is a fallback for official labels outside the defined taxonomy — not the automatic destination for an unmapped niche tag.

This preserves the original dataset where it behaves sensibly while correcting systematic problems.

In [8]:
FINAL_GENRES = [
    "Pop",
    "Indie",
    "Rock",
    "Hip-Hop",
    "R&B/Soul",
    "Country",
    "Electronic/Dance",
    "Folk",
    "Reggae/Dancehall",
    "Latin/Tropical",
    "Afrobeats/African Pop",
    "Jazz",
    "Classical/Musical",
    "Blues",
    "Other",
]

BASE_GENRE_MAP = {
    "Pop": "Pop",
    "Rock": "Rock",
    "Hip-Hop": "Hip-Hop",
    "R&B": "R&B/Soul",
    "Country": "Country",
    "Electronic": "Electronic/Dance",
    "Folk": "Folk",
    "Jazz": "Jazz",
    "Classical": "Classical/Musical",
    "Blues": "Blues",
}

songs["genre_original"] = songs["genre"]

songs["genre_base"] = (
    songs["genre_original"]
    .map(BASE_GENRE_MAP)
    .fillna("Other")
)

## 7. High-confidence niche-tag votes

These tags provide enough musical information to vote for one of the final broad categories.

A few deliberate choices:

- generic/geographic `indie` → **Indie**
- `indie rock` → **Rock**
- `indie r&b` → **R&B/Soul**
- `indie pop` → **Pop**
- `ska` → **Reggae/Dancehall**
- `ska punk` → **Rock**
- `pop punk` → **Rock**
- `synthpop` → **Electronic/Dance**
- generic `industrial`, `darkwave`, `new wave`, and `riddim` do **not** hard-override because they are genuinely cross-genre in this dataset.
- thematic/context tags such as `christian`, `christmas`, and `worship` do not vote for a primary musical genre.

In [9]:
GENRE_VOTE_TAGS = {

    "Reggae/Dancehall": {
        "calypso",
        "christian reggae",
        "dancehall",
        "dub",
        "lovers rock",
        "nz reggae",
        "ragga",
        "reggae",
        "reggae en español",
        "reggae pop",
        "reggae rock",
        "rocksteady",
        "roots reggae",
        "ska",
        "soca",
    },

    "Afrobeats/African Pop": {
        "afro adura",
        "afro r&b",
        "afro soul",
        "afrobeat",
        "afrobeats",
        "afrogospel",
        "afropiano",
        "afropop",
        "afroswing",
        "alté",
        "amapiano",
        "asakaa",
        "azonto",
        "bongo flava",
        "ghanaian hip hop",
        "highlife",
        "hiplife",
        "nigerian drill",
        "rumba congolaise",
    },

    "Latin/Tropical": {
        "bachata",
        "bolero",
        "bossa nova",
        "brazilian funk",
        "chilean mambo",
        "christian reggaeton",
        "corridos bélicos",
        "corridos tumbados",
        "cumbia",
        "cumbia norteña",
        "cumbia sonidera",
        "electro corridos",
        "electrocumbia",
        "forró",
        "forró tradicional",
        "funk carioca",
        "latin",
        "latin afrobeats",
        "latin alternative",
        "latin country",
        "latin dance",
        "latin folk",
        "latin folklore",
        "latin hip hop",
        "latin house",
        "latin indie",
        "latin jazz",
        "latin pop",
        "latin r&b",
        "latin rock",
        "mambo",
        "mariachi",
        "merengue",
        "mpb",
        "música mexicana",
        "new mpb",
        "norteño",
        "norteño-sax",
        "pagode",
        "pagode baiano",
        "ranchera",
        "reggaeton",
        "reggaeton chileno",
        "reggaeton mexa",
        "salsa",
        "samba",
        "sertanejo",
        "sertanejo tradicional",
        "sertanejo universitário",
        "tango",
        "trap latino",
        "urbano latino",
    },

    "Indie": {
        "indie",
        "chinese indie",
        "german indie",
        "hindi indie",
        "indian indie",
        "indie québécois",
        "indonesian indie",
        "japanese indie",
        "lo-fi indie",
        "mexican indie",
        "pinoy indie",
        "taiwanese indie",
        "tamil indie",
        "vietnam indie",
    },

    "R&B/Soul": {
        "alternative r&b",
        "classic soul",
        "dark r&b",
        "funk",
        "indie r&b",
        "indie soul",
        "motown",
        "neo soul",
        "new jack swing",
        "northern soul",
        "philly soul",
        "quiet storm",
        "r&b",
        "retro soul",
        "soul",
        "trap soul",
    },

    "Hip-Hop": {
        "alternative hip hop",
        "boom bap",
        "cloud rap",
        "crunk",
        "drill",
        "east coast hip hop",
        "emo rap",
        "experimental hip hop",
        "g-funk",
        "gangster rap",
        "hardcore hip hop",
        "hip hop",
        "melodic rap",
        "old school hip hop",
        "rap",
        "southern hip hop",
        "trap",
        "underground hip hop",
        "west coast hip hop",
    },

    "Electronic/Dance": {
        "deep house",
        "disco",
        "drum and bass",
        "dubstep",
        "ebm",
        "edm",
        "electro house",
        "electronic",
        "eurodance",
        "future bass",
        "hardstyle",
        "house",
        "melodic bass",
        "progressive house",
        "synthpop",
        "tech house",
        "techno",
        "trance",
        "tropical house",
    },

    "Country": {
        "alt country",
        "americana",
        "bluegrass",
        "classic country",
        "country",
        "honky tonk",
        "outlaw country",
        "red dirt",
        "texas country",
        "traditional country",
    },

    "Jazz": {
        "bebop",
        "big band",
        "cool jazz",
        "jazz",
        "jazz funk",
        "jazz fusion",
        "smooth jazz",
        "soul jazz",
        "swing music",
        "vocal jazz",
    },

    "Blues": {
        "blues",
        "classic blues",
        "country blues",
        "delta blues",
        "modern blues",
    },

    "Classical/Musical": {
        "baroque",
        "chamber music",
        "choral",
        "classical",
        "classical crossover",
        "contemporary classical",
        "musicals",
        "opera",
        "orchestral",
        "renaissance",
    },

    "Folk": {
        "anti-folk",
        "celtic folk",
        "folk",
        "folk pop",
        "indie folk",
        "singer-songwriter",
        "traditional folk",
    },

    "Rock": {
        "alternative rock",
        "art rock",
        "blues rock",
        "classic rock",
        "death metal",
        "funk rock",
        "garage rock",
        "glam rock",
        "grunge",
        "hard rock",
        "hardcore",
        "indie rock",
        "industrial metal",
        "industrial rock",
        "metal",
        "metalcore",
        "pop punk",
        "post-grunge",
        "post-hardcore",
        "power metal",
        "progressive rock",
        "psychedelic rock",
        "psychobilly",
        "punk",
        "rock",
        "rock and roll",
        "ska punk",
        "skate punk",
        "soft rock",
        "yacht rock",
    },

    "Pop": {
        "art pop",
        "baroque pop",
        "bedroom pop",
        "dance pop",
        "electropop",
        "funk pop",
        "indie pop",
        "j-pop",
        "k-pop",
        "pop",
        "power pop",
        "soft pop",
    },
}

# Additional explicit genre tags discovered during audit

GENRE_VOTE_TAGS["Rock"].update({
    "christian rock",
    "christian alternative rock",
    "southern rock",
    "celtic rock",
    "j-rock",
    "krautrock",
})

GENRE_VOTE_TAGS["Country"].update({
    "country christian",
    "christian bluegrass",
    "southern gospel",
})

GENRE_VOTE_TAGS["Folk"].update({
    "christian folk",
    "celtic",
})

GENRE_VOTE_TAGS["Pop"].update({
    "christian pop",
    "pop worship",
    "dream pop",
    "acoustic pop",
})

GENRE_VOTE_TAGS["R&B/Soul"].update({
    "christian r&b",
    "uk r&b",
})

GENRE_VOTE_TAGS["Hip-Hop"].update({
    "christian hip hop",
    "grime",
    "uk grime",
    "chinese hip hop",
    "k-rap",
    "miami bass",
})

GENRE_VOTE_TAGS["Electronic/Dance"].update({
    "alternative dance",
    "electro swing",
    "synthwave",
    "witch house",
})

GENRE_VOTE_TAGS["Jazz"].update({
    "acid jazz",
    "experimental jazz",
})

GENRE_VOTE_TAGS["Classical/Musical"].update({
    "gregorian chant",
})

# Tags that are meaningful but too cross-genre to force a primary classification.

INHERIT_TAGS = {
    "industrial",
    "darkwave",
    "new wave",
    "riddim",
    "folk rock",
    "country rock",
    "gospel",
    "traditional gospel",
}

# Descriptors/context tags that should not vote for a primary musical genre.

PRIMARY_GENRE_IGNORE_TAGS = {
    "christian",
    "christmas",
    "worship",
    "ccm",
    "hymns",
    "soundtrack",
    "children's music",
    "comedy",
}

### Sanity check: a niche tag should not vote for multiple categories

This catches accidental duplicates in the manual vote sets before classification runs.

In [10]:
tag_to_categories = {}

for category, tags in GENRE_VOTE_TAGS.items():
    for tag in tags:
        tag_to_categories.setdefault(tag, []).append(category)

duplicate_vote_tags = {
    tag: categories
    for tag, categories in tag_to_categories.items()
    if len(categories) > 1
}

assert not duplicate_vote_tags, (
    "Some niche tags vote for multiple categories: "
    f"{duplicate_vote_tags}"
)

print(
    "Unique high-confidence voting tags:",
    len(tag_to_categories),
)

Unique high-confidence voting tags: 268


## 8A. Secondary thematic features

Some niche tags describe themes, communities, or contexts rather than a
primary musical genre.

Religious tags therefore do not determine `genre_engineered`, but are
preserved as a separate binary feature.

This allows a song to be, for example:

- `genre_engineered = "Hip-Hop"`
- `religious_music = 1`

rather than forcing "Christian" to become a primary genre.

In [11]:
RELIGIOUS_TAGS = {
    "christian",
    "christian alternative rock",
    "christian bluegrass",
    "christian folk",
    "christian hip hop",
    "christian jazz",
    "christian lo-fi",
    "christian pop",
    "christian r&b",
    "christian reggae",
    "christian reggaeton",
    "christian rock",
    "country christian",
    "ccm",
    "gospel",
    "traditional gospel",
    "southern gospel",
    "afrogospel",
    "brazilian gospel",
    "funk gospel",
    "pagode gospel",
    "sertanejo gospel",
    "worship",
    "pop worship",
    "hymns",
}

vote_tags = set().union(*GENRE_VOTE_TAGS.values())

rule_conflicts = {
    "vote_and_inherit": vote_tags & INHERIT_TAGS,
    "vote_and_ignore": vote_tags & PRIMARY_GENRE_IGNORE_TAGS,
    "inherit_and_ignore": INHERIT_TAGS & PRIMARY_GENRE_IGNORE_TAGS,
}

rule_conflicts = {
    name: tags
    for name, tags in rule_conflicts.items()
    if tags
}

assert not rule_conflicts, (
    f"Conflicting genre rules found: {rule_conflicts}"
)

print("No conflicting genre rules.")

No conflicting genre rules.


In [12]:
def has_religious_tag(tags):
    tag_set = {
        str(tag).lower().strip()
        for tag in tags
        if pd.notna(tag)
    }

    return int(bool(tag_set & RELIGIOUS_TAGS))


songs["religious_music"] = (
    songs["niche_genre_list"]
    .apply(has_religious_tag)
)

In [13]:
print(
    songs["religious_music"]
    .value_counts(dropna=False)
)

print(
    f"Religiously tagged songs: "
    f"{songs['religious_music'].sum():,}"
)

religious_music
0    520340
1     30282
Name: count, dtype: int64
Religiously tagged songs: 30,282


In [14]:
display(
    songs.loc[
        songs["religious_music"] == 1,
        [
            "name",
            "artists",
            "genre_original",
            "genre_engineered",
            "niche_genres",
        ]
    ].sample(30, random_state=42)
)

KeyError: "['genre_engineered'] not in index"

## 8. Score niche tags and apply overrides

Every exact tag match contributes one vote to its broad category.

Why exact matching? Substring matching causes false positives such as:

- `ska` matching `skate punk`
- `dub` matching `dubstep`
- `rap` matching `trap`

If multiple categories tie for the highest score, the notebook keeps the mapped official genre rather than forcing a questionable override.

In [ ]:
def genre_scores(tags):
    tag_set = {
        str(tag).lower().strip()
        for tag in tags
        if pd.notna(tag)
    }

    return {
        category: len(tag_set & category_tags)
        for category, category_tags in GENRE_VOTE_TAGS.items()
    }


def choose_genre_override(tags):
    scores = genre_scores(tags)

    best_score = max(scores.values())

    if best_score == 0:
        return None

    winners = [
        category
        for category, score in scores.items()
        if score == best_score
    ]

    # Ambiguous niche evidence: keep official/base genre.
    if len(winners) != 1:
        return None

    return winners[0]

In [ ]:
songs["genre_override"] = (
    songs["niche_genre_list"]
    .apply(choose_genre_override)
)

songs["genre_engineered"] = (
    songs["genre_override"]
    .fillna(songs["genre_base"])
)

unexpected_genres = (
    set(songs["genre_engineered"].dropna().unique())
    - set(FINAL_GENRES)
)

assert not unexpected_genres, (
    f"Unexpected engineered genres: {unexpected_genres}"
)

## 9. Audit the engineered result

Keep an audit trail rather than replacing the source columns.

Useful columns:

- `genre_original`: source dataset's broad genre
- `genre_base`: source genre renamed into the final taxonomy
- `niche_genres`: original niche-tag string
- `genre_override`: niche-tag winner, if there was one
- `genre_engineered`: final category used for analysis

In [ ]:
genre_summary = (
    songs["genre_engineered"]
    .value_counts(dropna=False)
    .rename_axis("genre_engineered")
    .reset_index(name="song_count")
)

display(genre_summary)

In [ ]:
changed_genres = songs.loc[
    songs["genre_engineered"] != songs["genre_base"],
    [
        "name",
        "artists",
        "genre_original",
        "genre_base",
        "niche_genres",
        "genre_override",
        "genre_engineered",
    ],
]

print(f"Rows changed by niche-tag overrides: {len(changed_genres):,}")

if len(changed_genres) > 0:
    display(
        changed_genres.sample(
            min(40, len(changed_genres)),
            random_state=42,
        )
    )

## 10. Tag-rule coverage audit

There are 754 unique niche tags in this dataset, but they do **not** all need a manual category.

This audit distinguishes:

- `vote` — explicitly supplies evidence for a category
- `inherit` — deliberately cross-genre; use the official/base genre unless other tags resolve it
- `ignore` — thematic/contextual; does not vote
- `fallback_to_official` — not manually mapped because the base broad genre remains the default

### `vote`

- This niche tag actively contributes a vote toward a primary genre.
- `manual_vote_category` is populated.

### `inherit`

- Deliberately cross-genre.
- This tag itself does not vote.
- Another tag on the same song can still cause an override.
- Otherwise, keep `genre_base`.

### `ignore`

- Deliberately not a primary-genre descriptor.
- Examples: `christian`, `christmas`, `worship`.
- Does not vote, but can still be used for a separate feature.

### `fallback_to_official`

- We have not assigned this niche tag a manual rule.
- It does not vote.
- If no other tag votes, keep `genre_base`.

This means every niche tag is covered by the **classification logic**, even though only a subset requires an explicit override.

In [ ]:
all_niche_tags = set(
    songs["niche_genre_list"]
    .explode()
    .dropna()
    .astype(str)
    .str.lower()
    .str.strip()
)

vote_tags = set(tag_to_categories)

def get_rule_type(tag):
    if tag in vote_tags:
        return "vote"
    if tag in INHERIT_TAGS:
        return "inherit"
    if tag in PRIMARY_GENRE_IGNORE_TAGS:
        return "ignore"
    return "fallback_to_official"


tag_rule_summary = pd.Series(
    [get_rule_type(tag) for tag in all_niche_tags]
).value_counts()

print("Total niche genres:", len(all_niche_tags))
display(tag_rule_summary.rename("tag_count"))

## 11. Optional: inspect the saved 754-tag audit

The previously generated `genre_tag_audit.csv` is useful for reviewing high-volume, low-agreement tags without rebuilding the file every time.

Low agreement is a **review flag**, not proof that a tag needs an override. Some tags genuinely span genres.

In [ ]:
GENRE_AUDIT_PATH = DATA_PROCESSED / "genre_tag_audit.csv"

if GENRE_AUDIT_PATH.exists():
    genre_audit = pd.read_csv(GENRE_AUDIT_PATH)

    review_candidates = (
        genre_audit[
            (genre_audit["total_count"] >= 500)
            & (genre_audit["dominant_share"] < 0.80)
        ]
        .copy()
    )

    review_candidates["rule_type"] = (
        review_candidates["niche_genre"]
        .map(get_rule_type)
    )

    review_candidates["manual_vote_category"] = (
        review_candidates["niche_genre"]
        .map(
            {
                tag: category
                for category, tags in GENRE_VOTE_TAGS.items()
                for tag in tags
            }
        )
    )

    display(
        review_candidates[
            [
                "niche_genre",
                "total_count",
                "dominant_genre",
                "dominant_share",
                "second_genre",
                "second_share",
                "rule_type",
                "manual_vote_category",
                "examples",
            ]
        ].sort_values("total_count", ascending=False)
    )

else:
    print(
        "genre_tag_audit.csv not found. "
        "The rebuild code is preserved in the appendix."
    )

## 12. Save the engineered metadata

This saved file becomes the lightweight song-level lookup table for the matching stage.

The original `songs.csv` remains untouched.

In [ ]:
ENGINEERED_METADATA_PATH = (
    DATA_PROCESSED / "song_metadata_engineered.parquet"
)

songs.to_parquet(
    ENGINEERED_METADATA_PATH,
    index=False,
)

print("Saved:", ENGINEERED_METADATA_PATH)

# Next step — Billboard ↔ 550K song matching

The genre work should now be treated as a **frozen first-pass taxonomy** unless the audit reveals a serious systematic error.

The next substantive step is matching each unique Billboard `Song` + `Artist` pair to the 550K song table, obtaining a canonical song ID, and then merging features back to the full weekly Billboard chart.

The matching code below is preserved but commented out so it does not run before you are ready.

In [ ]:
# --- NEXT STEP: text normalization for exact matching ---
#
# def normalize_text(value):
#     if pd.isna(value):
#         return ""
#
#     value = str(value).lower().strip()
#     value = unicodedata.normalize("NFKD", value)
#     value = "".join(
#         char
#         for char in value
#         if not unicodedata.combining(char)
#     )
#     value = value.replace("&", "and")
#     value = re.sub(r"[^\w\s]", " ", value)
#     value = re.sub(r"\s+", " ", value)
#
#     return value.strip()
#
#
# billboard_songs["title_key"] = (
#     billboard_songs["Song"]
#     .map(normalize_text)
# )
#
# billboard_songs["artist_key"] = (
#     billboard_songs["Artist"]
#     .map(normalize_text)
# )
#
# # The 550K `artists` field is itself a stringified list.
# def parse_artist_list(value):
#     if isinstance(value, list):
#         return value
#
#     if pd.isna(value):
#         return []
#
#     try:
#         parsed = ast.literal_eval(value)
#         return parsed if isinstance(parsed, list) else []
#
#     except (ValueError, SyntaxError, TypeError):
#         return []
#
#
# songs["artist_list"] = (
#     songs["artists"]
#     .apply(parse_artist_list)
# )
#
# songs["primary_artist"] = (
#     songs["artist_list"]
#     .str[0]
# )
#
# songs["title_key"] = (
#     songs["name"]
#     .map(normalize_text)
# )
#
# songs["artist_key"] = (
#     songs["primary_artist"]
#     .map(normalize_text)
# )

# Appendix — preserved thought process / retired exploration

These cells are intentionally commented out. They document how the current decisions were reached but are no longer required for the main pipeline.

In [ ]:
# --- Historical exploration: previewing the huge raw CSV ---
#
# song_preview = pd.read_csv(
#     SONGS_PATH,
#     nrows=5,
# )
#
# print("Columns:", song_preview.columns.tolist())
# display(song_preview)

In [ ]:
# --- Historical exploration: suspicious release years ---
#
# songs["year"].value_counts().sort_index()
#
# print("Earliest year:", songs["year"].min())
# print("Latest year:", songs["year"].max())
#
# # This revealed obvious errors such as modern songs assigned to 1900.
# # Decision: do not use the 550K `year` field as the forecasting timeline.
#
# display(
#     songs.loc[
#         songs["year"] < 1920,
#         ["name", "artists", "year", "genre", "niche_genres"],
#     ]
# )

In [ ]:
# --- Historical exploration: auditing the source Folk category ---
#
# display(
#     songs.loc[
#         songs["genre"] == "Folk",
#         ["name", "artists", "genre", "niche_genres"],
#     ]
# )
#
# # This is where systematic reggae/dancehall misclassification became obvious.

In [ ]:
# --- Historical exploration: official genre + full niche-combination counts ---
#
# genre_combo_counts = (
#     songs
#     .groupby(
#         ["genre", "niche_genres"],
#         dropna=False,
#     )
#     .size()
#     .reset_index(name="count")
#     .sort_values("count", ascending=False)
# )
#
# display(genre_combo_counts.head(50))
#
# genre_combo_counts.to_csv(
#     DATA_PROCESSED / "genre_combo_counts.csv",
#     index=False,
# )

In [ ]:
# --- Historical exploration: initial "covered vs uncovered" override audit ---
#
# # Important lesson:
# # "Not explicitly covered by an override" does NOT mean "Other."
# # Unmapped niche tags fall back to the mapped official broad genre.
#
# override_tags = set().union(*GENRE_VOTE_TAGS.values())
#
# covered_tags = all_niche_tags & override_tags
# uncovered_tags = all_niche_tags - override_tags
#
# print("Total niche genres:", len(all_niche_tags))
# print("Covered by explicit votes:", len(covered_tags))
# print("Using fallback/inherit/ignore logic:", len(uncovered_tags))

In [ ]:
# --- Historical exploration: dominant official genre by niche tag ---
#
# tag_genre_counts = (
#     songs[["genre_original", "niche_genre_list"]]
#     .explode("niche_genre_list")
#     .dropna(subset=["niche_genre_list"])
#     .assign(
#         niche_genre=lambda x:
#         x["niche_genre_list"]
#         .str.lower()
#         .str.strip()
#     )
#     .groupby(["niche_genre", "genre_original"])
#     .size()
#     .reset_index(name="count")
# )
#
# tag_totals = (
#     tag_genre_counts
#     .groupby("niche_genre")["count"]
#     .sum()
#     .rename("total")
# )
#
# dominant_genres = (
#     tag_genre_counts
#     .sort_values("count", ascending=False)
#     .drop_duplicates("niche_genre")
#     .merge(tag_totals, on="niche_genre")
# )
#
# dominant_genres["dominant_share"] = (
#     dominant_genres["count"]
#     / dominant_genres["total"]
# )
#
# # Practical review screen used during exploration:
# # high-volume tags + dominant share below 80%.
# # Low agreement alone does NOT mean the tag is wrong.

In [ ]:
# --- Historical exploration: rebuilding genre_tag_audit.csv ---
#
# exploded_genres = (
#     songs[
#         [
#             "name",
#             "artists",
#             "genre_original",
#             "niche_genre_list",
#         ]
#     ]
#     .explode("niche_genre_list")
#     .dropna(subset=["niche_genre_list"])
#     .rename(columns={"niche_genre_list": "niche_genre"})
# )
#
# exploded_genres["niche_genre"] = (
#     exploded_genres["niche_genre"]
#     .str.lower()
#     .str.strip()
# )
#
# genre_distribution = (
#     exploded_genres
#     .groupby(["niche_genre", "genre_original"])
#     .size()
#     .unstack(fill_value=0)
# )
#
# genre_shares = genre_distribution.div(
#     genre_distribution.sum(axis=1),
#     axis=0,
# )
#
# genre_distribution.columns = [
#     f"{col}_count"
#     for col in genre_distribution.columns
# ]
#
# genre_shares.columns = [
#     f"{col}_share"
#     for col in genre_shares.columns
# ]
#
# genre_audit = pd.concat(
#     [genre_distribution, genre_shares],
#     axis=1,
# )
#
# genre_audit["total_count"] = (
#     genre_distribution.sum(axis=1)
# )
#
# official_counts = (
#     exploded_genres
#     .groupby(["niche_genre", "genre_original"])
#     .size()
#     .reset_index(name="count")
# )
#
# official_counts["rank"] = (
#     official_counts
#     .groupby("niche_genre")["count"]
#     .rank(method="first", ascending=False)
# )
#
# top1 = (
#     official_counts[official_counts["rank"] == 1]
#     .set_index("niche_genre")
# )
#
# top2 = (
#     official_counts[official_counts["rank"] == 2]
#     .set_index("niche_genre")
# )
#
# genre_audit["dominant_genre"] = top1["genre_original"]
# genre_audit["dominant_share"] = (
#     top1["count"] / genre_audit["total_count"]
# )
#
# genre_audit["second_genre"] = top2["genre_original"]
# genre_audit["second_share"] = (
#     top2["count"] / genre_audit["total_count"]
# )
#
#
# def get_examples(group, n=3):
#     examples = (
#         group[["name", "artists"]]
#         .drop_duplicates()
#         .head(n)
#     )
#
#     return " | ".join(
#         examples["name"].astype(str)
#         + " — "
#         + examples["artists"].astype(str)
#     )
#
#
# examples = (
#     exploded_genres
#     .groupby("niche_genre")[["name", "artists"]]
#     .apply(get_examples)
#     .rename("examples")
# )
#
# genre_audit = (
#     genre_audit
#     .join(examples)
#     .reset_index()
#     .sort_values("total_count", ascending=False)
# )
#
# genre_audit.to_csv(
#     DATA_PROCESSED / "genre_tag_audit.csv",
#     index=False,
# )

## Current stopping point

At this stage the notebook has:

- a clean Billboard popularity backbone,
- a memory-conscious 550K metadata load,
- documented rejection of the unreliable `year` field,
- a reproducible 15-category genre taxonomy,
- explicit high-confidence niche-tag overrides,
- safe fallback behavior for unmapped/ambiguous tags,
- audit columns preserving every classification decision,
- and commented matching code ready for the next stage.

**Recommended next action:** run the cleaned notebook through the genre audit, inspect a sample of changed rows, then freeze genre engineering and move to exact Billboard matching.